# Практическая работа № 2. Работа с LLM через API

**Студент:** Архангельский Данила Андреевич  
**Группа:** ИСТ-05  
**Вариант:** 01 (`rental_01.csv`)


In [ ]:
# При необходимости: %pip install -U langchain-gigachat pandas pydantic python-dotenv
import os, csv
from pathlib import Path
from typing import Optional
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

load_dotenv()
API_KEY = os.getenv('GIGA_KEY')
if not API_KEY:
    raise RuntimeError('Добавьте новый GIGA_KEY в локальный .env рядом с ноутбуком')
llm = GigaChat(credentials=API_KEY, model='GigaChat-2', verify_ssl_certs=False, temperature=0.2, max_tokens=1000)
print(llm.invoke('Ответь одним словом: работает?').content)


In [ ]:
VARIANT = '01'
def load_rental(path):
    rows=[]
    with open(path, encoding='utf-8-sig', newline='') as f:
        reader=csv.reader(f, delimiter=';', quotechar='"')
        next(reader, None)
        for row in reader:
            if row: rows.append({'amount': int(row[0]), 'text': ';'.join(row[1:]).strip()})
    return pd.DataFrame(rows)
df = load_rental(Path('data') / f'rental_{VARIANT}.csv').head(15).copy().reset_index(drop=True)
assert len(df)==15
df.head(15)


## Часть 1. Базовый промпт и Few-shot


In [ ]:
basic_prompt = PromptTemplate(input_variables=['text'], template='Определи общее число проживающих, учитывай взрослых и детей, не считай животных. Если данных недостаточно, верни 0. Верни только целое число.\nЗаявка: {text}\nКоличество:')
basic_chain = basic_prompt | llm | StrOutputParser()
system = SystemMessagePromptTemplate.from_template('Извлеки общее число проживающих. Считай взрослых и детей, не считай животных. Примеры: «пара»=2; «один взрослый и ребёнок»=2; «2 взрослых и 2 детей»=4; «с женой и двумя детьми»=4. Если однозначно определить нельзя, верни 0. Только целое число.')
human = HumanMessagePromptTemplate.from_template('Заявка: {text}\nКоличество проживающих:')
few_chain = ChatPromptTemplate.from_messages([system,human]) | llm | StrOutputParser()
def run_chain(chain, texts):
    out=[]
    for text in texts:
        try: out.append(chain.invoke({'text':text}))
        except Exception as e: out.append(f'ERROR: {type(e).__name__}: {e}')
    return out
def to_num(s): return pd.to_numeric(s.astype(str).str.strip(), errors='coerce')
df['result_basic']=run_chain(basic_chain, df.text)
df['result_fewshot']=run_chain(few_chain, df.text)
for col in ['result_basic','result_fewshot']:
    pred=to_num(df[col]); correct=int(pred.eq(df.amount).sum())
    print(f'{col}: {correct}/{len(df)}; ошибок {len(df)-correct}; точность {correct/len(df):.1%}')
errors=df.loc[to_num(df.result_basic).ne(df.amount) | to_num(df.result_fewshot).ne(df.amount), ['text','amount','result_basic','result_fewshot']]
errors


## Часть 2. Извлечение дополнительных полей в JSON


In [ ]:
class RentalRequest(BaseModel):
    count_adults: Optional[int] = Field(default=None, description='Количество взрослых')
    count_children: Optional[int] = Field(default=None, description='Количество детей')
    start_date: Optional[str] = Field(default=None, description='Самая ранняя дата заезда, ДД.ММ')
    nights: Optional[int] = Field(default=None, description='Количество ночей; для диапазона максимум')
    price_per_day: Optional[int] = Field(default=None, description='Цена за сутки; для диапазона максимум')
    remarks: Optional[str] = Field(default=None, description='Особые пожелания, животные, удобства')
parser=JsonOutputParser(pydantic_object=RentalRequest)
sys=SystemMessagePromptTemplate.from_template('Извлеки поля из заявки. Не выдумывай отсутствующее: null. start_date — самая ранняя дата; nights и price_per_day — максимум диапазона. remarks — пожелания, животные и удобства. Верни JSON строго по схеме. Пример: «2 взрослых и ребёнок, с 10.08 по 15.08, до 3000 руб/сутки, бассейн» => {"count_adults":2,"count_children":1,"start_date":"10.08","nights":5,"price_per_day":3000,"remarks":"бассейн"}')
usr=HumanMessagePromptTemplate.from_template('Заявка: {text}\n{format_instructions}')
prompt=ChatPromptTemplate.from_messages([sys,usr]).partial(format_instructions=parser.get_format_instructions())
chain=prompt | llm | parser
extracted=[]
for text in df.text:
    try: extracted.append(RentalRequest(**chain.invoke({'text':text})).model_dump())
    except Exception as e: extracted.append({'_error':f'{type(e).__name__}: {e}'})
ext=pd.DataFrame(extracted)
for f in RentalRequest.model_fields:
    if f not in ext: ext[f]=None
results=pd.concat([df[['text','amount']],ext],axis=1)
results.to_csv(f'rental_{VARIANT}_extracted.csv',index=False,encoding='utf-8-sig')
results


## Ручная разметка и оценка качества
Пустые значения в эталонном файле означают, что поле не указано однозначно. `remarks` оценивается качественно; средняя метрика рассчитывается по пяти числовым/скалярным полям.


In [ ]:
gold=pd.read_csv(f'gold_fields_{VARIANT}.csv',dtype=str).fillna('')
assert len(gold)==len(df) and gold.text.equals(df.text.astype(str)), 'Эталон не совпадает с исходными заявками'
fields=['count_adults','count_children','start_date','nights','price_per_day']
def norm(x):
    if pd.isna(x) or str(x).strip().lower() in {'','nan','none','null'}: return ''
    s=str(x).strip(); return s[:-2] if s.endswith('.0') else s
scores={}
for field in fields:
    matches=sum(norm(a)==norm(b) for a,b in zip(ext[field],gold[field]))
    scores[field]=matches/len(gold)
    print(f'{field}: {matches}/{len(gold)} = {scores[field]:.1%}')
print(f'Средняя точность: {sum(scores.values())/len(scores):.1%}')
print('remarks: сравнить содержательно вручную')


## Выводы
После выполнения всех ячеек зафиксируйте реальные значения точности базового промпта, Few-shot и каждого дополнительного поля. Выберите пример ошибки из `errors` и объясните причину. Не подставляйте предполагаемые метрики: они зависят от фактического ответа API.
